In [0]:
#Imports
from pyspark.sql.functions import from_json, col, regexp_replace, trim, explode
from pyspark.sql.functions import split
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StructType, StructField, StringType

In [0]:
# Project setup: makes the src/ package importable and loads the config
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from src.utils.config_loader import load_config
from src.utils.spark_helpers import run_sql_file

cfg = load_config()

In [0]:
Games=spark.table("steam_data.default.steam_games")

Review=spark.table("steam_data.default.review")

In [0]:
def table_size(df, name):
    data = [(name, df.count(), len(df.columns))]
    spark.createDataFrame(data, ["Table", "Rows", "Cols"]).show()
table_size(Games, "Games")
table_size(Review, "Review")

In [0]:
%sql
select * from steam_data.default.steam_games

Drop Duplicates


In [0]:
Games=Games.dropDuplicates()
Review=Review.dropDuplicates()

print("AFTER DROPPING DUPLICATES")

table_size(Games,"Games")
table_size(Review,"Reviews")




##Checking for null

In [0]:

null_count={}
for colu in Games.columns:   
    number_of_null = Games.filter(col(colu).isNull()).count()
    null_count[colu]=number_of_null

null_count_df = spark.createDataFrame(null_count.items(), ["column", "null_count"])
null_count_df.show()


Selecting the columns suited for our business problem

In [0]:
columns_to_keep=['app_id','name','release_date','price','price_status','estimated_owners','genres','categories','positive','negative','recommendations','peak_ccu','metacritic_score','user_score','median_playtime_forever','windows','mac','linux',]
Games=Games.select(columns_to_keep)
# Games=Games.withColumnRenamed("supported_Languages","supported_languages")
Games=Games.withColumn("price",Games["price"].cast("double"))

In [0]:
# Games.describe().show()

display(Games)

Handlin the estimated owner column

In [0]:
Games = Games.withColumn(
    "estimated_owners",
    split(col("estimated_owners"), r"\.\.|\s*-\s*")
)

Games = Games.withColumn(
    "estimated_owners",
    (
        regexp_replace(trim(col("estimated_owners").getItem(0)), ",", "").cast("double")
        +
        regexp_replace(trim(col("estimated_owners").getItem(1)), ",", "").cast("double")
    ) / 2
)

In [0]:
Games.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("steam_data.default.silver_games")

##WORKING WITH SILVER TABLE


In [0]:
silver_games=spark.table("steam_data.default.silver_games")
silver_games.describe().show()
display(silver_games)

##Fixing the Price column


In [0]:
from pyspark.sql import functions as F

before = silver_games.count()

silver_games = silver_games.filter(F.col("price").isNull() | (F.col("price") <= 1000))

after = silver_games.count()
print(f"Dropped {before - after} rows with price > 1000")

In [0]:
run_sql_file(spark, "silver/01_filter_price.sql", cfg)

The below visualisation makes it clear that most of the prices are below 100. There are some above 100. Anything above 200 looks unsual but not necessarily anomaly . So, I chekced steam and it seems that their most expensive game is 1000 dollars so anything greater than 1000 can be dropped.


In [0]:
%sql
select price,  count(app_id) as count from steam_data.default.silver_games
GROUP BY price

Databricks visualization. Run in Databricks to view.

##Handling Genres and Categories    

In [0]:
obj_schema = ArrayType(StructType([
    StructField("id", StringType(), True),
    StructField("description", StringType(), True)
]))
str_schema = ArrayType(StringType())

genre_is_json = col("genres").contains('"id"')

genre_json_rows = silver_games.filter(genre_is_json).withColumn("genres_parsed", from_json(col("genres"), obj_schema))
genre_str_rows  = silver_games.filter(~genre_is_json).withColumn("genres_parsed", from_json(col("genres"), str_schema))

print("JSON-format games      :", genre_json_rows.count())
print("Plain-list-format games:", genre_str_rows.count())

Basically , I split the genres which were given in the form of lists and formed a seperate table with app id and genre name so its a many to many relation

In [0]:
from pyspark.sql.functions import explode, trim, col

# JSON format: take the name from "description"
genre_json = (
    genre_json_rows
    .select("app_id", explode(col("genres_parsed")).alias("g"))
    .select("app_id", trim(col("g.description")).alias("genre"))
)

# Plain-list format: the element already is the name
genre_str = (
    genre_str_rows
    .select("app_id", explode(col("genres_parsed")).alias("genre"))
    .withColumn("genre", trim(col("genre")))
)

game_genre_df = (
    genre_json.unionByName(genre_str)
    .filter(col("genre").isNotNull() & (col("genre") != ""))
    .distinct()
)

game_genre_df.select("genre").distinct().orderBy("genre").show(100, truncate=False)  

In [0]:
game_genre_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("steam_data.default.game_genres")

One thing to observe there are 2 genres aventure and aventura and both looks like a spelling mistake of adventure. We will alter this.

In [0]:
run_sql_file(spark, "silver/02_fix_genre_spelling.sql", cfg)

Here there is one more problem . We can see some genres are Early access , Design and Illustration, Free to play etc. They don't really contribute to out analysis as they are not even the GAME GENRES we need. If a game has a genre of any of these type , we should drop it to imporve our analysis . One more issue could be that a game can have 2 or more genres. So an early access game can also be indie . We will only want indie to stay. We will drop all the apps which only has genres from the list we don't want .

Number of games which doesn't have genre from our list of interested genres

In [0]:
%sql
SELECT COUNT(*) AS games_to_delete
FROM (
    SELECT app_id
    FROM steam_data.default.game_genres
    GROUP BY app_id
    HAVING SUM(
        CASE
            WHEN genre IN (
                'Action',
                'Adventure',
                'Casual',
                'Indie',
                'Massively Multiplayer',
                'RPG',
                'Racing',
                'Simulation',
                'Sports',
                'Strategy'
            )
            THEN 1
            ELSE 0
        END
    ) = 0
);

%md
Delete games which doesn't have genre from our list of interested genres

In [0]:
run_sql_file(spark, "silver/03_delete_games_without_valid_genre.sql", cfg)

In [0]:
run_sql_file(spark, "silver/04_delete_invalid_genres.sql", cfg)

We will do same kind of preprocessing for categories

In [0]:
obj_schema = ArrayType(StructType([
    StructField("id", StringType(), True),
    StructField("description", StringType(), True)
]))
str_schema = ArrayType(StringType())

category_is_json = col("categories").contains('"id"')

category_json_rows = silver_games.filter(category_is_json).withColumn("categories_parsed", from_json(col("categories"), obj_schema))
category_str_rows  = silver_games.filter(~category_is_json).withColumn("categories_parsed", from_json(col("categories"), str_schema))

print("JSON-format games      :", category_json_rows.count())
print("Plain-list-format games:", category_str_rows.count())

In [0]:

# JSON format: take the name from "description"
category_json = (
    category_json_rows
    .select("app_id", explode(col("categories_parsed")).alias("g"))
    .select("app_id", trim(col("g.description")).alias("category"))
)

# Plain-list format: the element already is the name
category_str = (
    category_str_rows
    .select("app_id", explode(col("categories_parsed")).alias("category"))
    .withColumn("category", trim(col("category")))
)

game_category_df = (
    category_json.unionByName(category_str)
    .filter(col("category").isNotNull() & (col("category") != ""))
    .distinct()
)

game_category_df.select("category").distinct().orderBy("category").show(100, truncate=False)  

In [0]:
game_category_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("steam_data.default.game_categories")

Now we have our category and genre columns sorted. We can drop them from silver games to avoid redundancy 

In [0]:
run_sql_file(spark, "silver/05_enable_column_mapping.sql", cfg)

In [0]:
run_sql_file(spark, "silver/06_drop_genre_category_columns.sql", cfg)

In [0]:
%sql
select * from steam_data.default.silver_games

In [0]:
%sql
select genre,count(*) from steam_data.default.game_genres group by genre